# Bouquet systematics check — pinned σ=0 / pinned IDA-σ / production

Runs the bouquet **three times** to expose any inherent bias or systematic offset:

1. **j_phi pinned, σ=0** — every draw must reproduce the baseline (boundary ~0 mm, coil drift ~0 %). This is the strict no-systematic floor.
2. **j_phi pinned, IDA-like σ** — only the pressure profile perturbs; the response should be symmetric about the baseline.
3. **production** — free j_phi + IDA σ, the full bouquet.

After each mode we draw the same traces + coil-current diagnostics as the main example and print **signed** drift means — a non-zero mean is a bias; symmetric scatter (mean ≈ 0) is healthy.

> This notebook always regenerates (it is a validation run). It writes one `.h5` per mode and does not touch the golden draws.

## 1. Imports

In [ ]:
import sys, os

# --- TEMPORARY path-insert ----------------------------------------------------
# Until the updated backend is merged into a single shipped `bouquet`, run
# against the development worktree.  Remove this block once merged.
_DEV_BOUQUET = '/Users/danielburgess/Desktop/plasma/bouquet_coil_bounds'
sys.path[:] = [p for p in sys.path if not p.rstrip('/').endswith('/plasma/bouquet')
               and not p.rstrip('/').endswith('/plasma/bouquet_phantom_vsc')]
if _DEV_BOUQUET not in sys.path:
    sys.path.insert(0, _DEV_BOUQUET)
for _m in [m for m in list(sys.modules) if m == 'bouquet' or m.startswith('bouquet.')]:
    del sys.modules[_m]
# ------------------------------------------------------------------------------

import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

%matplotlib inline
%config InlineBackend.figure_format = "retina"

# --- TokaMaker ---
_OFT = '/Users/danielburgess/Desktop/plasma/OpenFUSIONToolkit/build_release'
sys.path.append(os.path.join(_OFT, 'python'))
from OpenFUSIONToolkit import OFT_env
from OpenFUSIONToolkit.TokaMaker import TokaMaker
from OpenFUSIONToolkit.TokaMaker.meshing import load_gs_mesh
from OpenFUSIONToolkit.TokaMaker.util import create_power_flux_fun

# --- Bouquet ---
import bouquet
assert 'bouquet_coil_bounds' in bouquet.__file__, f"wrong bouquet: {bouquet.__file__}"
from bouquet import (
    read_geqdsk, reconstruct_equilibrium, generate_bouquet,
    initialize_equilibrium_database, store_equilibrium,
    synthetic_ida_sigma,
    plot_bouquet, plot_traces, plot_boundary_point_traces,
    plot_geqdsk_bouquet, plot_pfile_bouquet, plot_coil_currents,
    plot_tokamaker_comparison, GEQDSKEquilibrium,
)
from bouquet import (filter_coil_currents, filter_boundaries,
                     select_indices, export_filtered)
from bouquet.io.pfile import read_pfile
print("bouquet:", bouquet.__file__)

## 2. Configuration

In [ ]:
import numpy as np
# ---- input files (shipped here) ----
GEQ  = 'D3Dlike_Hmode_baseline.geqdsk'
PF   = 'D3Dlike_Hmode_baseline.peqdsk'
MESH = 'DIIID_mesh.h5'

# ---- run controls ----
n_equils = 10        # draws per mode
seed     = 12345     # reproducible draws (modes 2 & 3 share the GPR sequence)

# ---- perturbation / GPR settings (identical to the main example) ----
pad_psi   = 1e-4
n_ls, t_ls, j_ls = 0.5, 0.4, 0.25
frac_jphi = 0.10
jBS_scale_range = [0.99, 1.01]
l_i_tolerance   = 0.05

# ---- coil-drift / homotopy / in-spec (fractions) ----
coil_drift      = 0.01
homotopy_passes = [(0.05, 0.10), (0.02, 0.05), (0.01, 0.01)]
inspec_F_max    = 0.02
inspec_VSC_max  = 0.02
vsc_soft_reg_weight = 1.0
p_thresh        = 0.05
ion_N, ion_Z, ion_A = [6, 1, 1], [6, 1, 1], [12, 2, 2]

# ---- the three modes: (header, sigma_scale, pin_jphi, label) ----
MODES = [
    ('D3Dlike_sys_pinned_s0',  0.0, True,  'pinned, sigma=0'),
    ('D3Dlike_sys_pinned_ida', 1.0, True,  'pinned, IDA sigma'),
    ('D3Dlike_sys_production', 1.0, False, 'production (free j_phi + IDA sigma)'),
]
print('modes:', [m[3] for m in MODES], ' n_equils =', n_equils)

## 3. Load baseline + kinetic profiles + σ envelopes

In [ ]:
eqdsk = read_geqdsk(GEQ, cocos=1)
psi_N = eqdsk.psi_N
F0 = abs(eqdsk.R_center * eqdsk.B_center)   # |F0| for the solve (sign handled by file)

pf = read_pfile(PF)
if pf.ion_species is None:
    pf.set_ion_species(N=ion_N, Z=ion_Z, A=ion_A)
pf.compute_quasineutrality()
psi_pf, Zeff = pf.compute_zeff()

# SI profiles on the equilibrium psi_N grid (for reconstruction)
ne_SI = interp1d(psi_pf, pf.ne*1e20, fill_value='extrapolate')(psi_N)   # m^-3
te_SI = interp1d(psi_pf, pf.te*1e3,  fill_value='extrapolate')(psi_N)   # eV
ni_SI = interp1d(psi_pf, pf.ni*1e20, fill_value='extrapolate')(psi_N)
ti_SI = interp1d(psi_pf, pf.ti*1e3,  fill_value='extrapolate')(psi_N)
Zeff_eq = np.clip(interp1d(psi_pf, Zeff, fill_value='extrapolate')(psi_N), 1.0, None)

print(f"Ip = {eqdsk.Ip:+.3e} A   Bt = {eqdsk.B_center:+.3f} T   F0 = {F0:.4f}")
print(f"ne0 = {ne_SI[0]:.3e} m^-3   Te0 = {te_SI[0]:.0f} eV   Ti0 = {ti_SI[0]:.0f} eV")

In [ ]:
# Kinetic-grid profiles + IDA-like fractional sigma envelopes (sigma_jphi is
# built per mode inside run_mode, since it scales recon's j_phi_fit).
ne_kin = pf.ne*1e20; te_kin = pf.te*1e3; ni_kin = pf.ni*1e20; ti_kin = pf.ti*1e3
sig_ne = synthetic_ida_sigma(psi_pf, 'ne') * ne_kin
sig_te = synthetic_ida_sigma(psi_pf, 'te') * te_kin
sig_ni = synthetic_ida_sigma(psi_pf, 'ni') * ni_kin
sig_ti = synthetic_ida_sigma(psi_pf, 'ti') * ti_kin
print('sigma envelopes built on', len(psi_pf), 'psi pts')

## 4. Initialize TokaMaker

In [ ]:
myOFT = OFT_env(nthreads=int(os.environ.get('NTHREADS', '4')))
mygs  = TokaMaker(myOFT)
mp, ml, mr, cd, cnd = load_gs_mesh(MESH)
mygs.setup_mesh(mp, ml, mr)
mygs.setup_regions(cond_dict=cnd, coil_dict=cd)
mygs.setup(order=3, F0=F0)
mygs.settings.maxits = 800
mygs.settings.pm = False
mygs.update_settings()
mygs.set_coil_vsc({'F9A': 1.0, 'F9B': -1.0})

# Weak regularization toward zero -> free inverse solve finds the coil currents
reg = [mygs.coil_reg_term({n: 1.0}, target=0.0, weight=1.0) for n in mygs.coil_sets]
reg.append(mygs.coil_reg_term({'#VSC': 1.0}, target=0.0, weight=1e-2))
mygs.set_coil_reg(reg_terms=reg)

iso = np.column_stack([eqdsk.boundary_R, eqdsk.boundary_Z])
isow = np.ones(len(iso)) * 200.0
mygs.set_isoflux(iso, weights=isow)
print(f"TokaMaker ready: {len(mp)} mesh pts, {len(mygs.coil_sets)} coil sets")

## 5. Helper: run one mode (reconstruct + generate) + summary

In [ ]:
def run_mode(header, sigma_scale, pin_jphi):
    """Fresh reconstruct (identical baseline every mode) + one bouquet."""
    mygs.set_isoflux(iso, weights=isow)
    guess = create_power_flux_fun(len(psi_N), 1.5, 1.5)['y']
    result = reconstruct_equilibrium(
        mygs, eqdsk, ne_SI, te_SI, ni_SI, ti_SI, Zeff_eq, iso, isow, pad_psi,
        guess_jinductive=guess, n_k=5, psi_bridge=0.99, rescale_j_BS=False,
        shelf_psi_N=0.0, initialize_psi=True)
    Ip_target  = abs(eqdsk.Ip)
    l_i_target = float(mygs.get_stats(lcfs_pad=pad_psi, li_normalization='std')['l_i'])
    sigma_jphi = frac_jphi * np.abs(result['j_phi_fit'])

    if os.path.exists(header + '.h5'):
        os.remove(header + '.h5')
    initialize_equilibrium_database(header)
    with open(GEQ, 'rb') as fh: geq_raw = fh.read()
    with open(PF,  'rb') as fh: pf_raw  = fh.read()
    mygs.set_isoflux(result['isoflux_pts'], weights=result['weights'])

    diag = generate_bouquet(
        mygs, psi_N, n_equils, header,
        result['j_phi_fit'],
        ne_kin, te_kin, ni_kin, ti_kin,
        sig_ne*sigma_scale, sig_te*sigma_scale,
        sig_ni*sigma_scale, sig_ti*sigma_scale, sigma_jphi*sigma_scale,
        n_ls, t_ls, j_ls,
        Ip_target, l_i_target, Zeff_eq,
        input_jinductive=result['j_inductive_fit'],
        l_i_tolerance=l_i_tolerance, psi_pad=pad_psi,
        constrain_sawteeth=False, recalculate_j_BS=True,
        jBS_scale_range=jBS_scale_range,
        pfile_bytes=pf_raw, baseline_eqdsk_bytes=geq_raw, baseline_pfile_bytes=pf_raw,
        diagnostic_plots=False, scan_val=0, psi_N_kinetic=psi_pf,
        coil_drift=coil_drift, homotopy_passes=homotopy_passes,
        inspec_F_max=inspec_F_max, inspec_VSC_max=inspec_VSC_max,
        p_thresh=p_thresh, vsc_soft_reg_weight=vsc_soft_reg_weight,
        save_truncate_eq=True, jphi_baseline=True, seed=seed, pin_jphi=pin_jphi,
    )
    print(f"  -> {len(diag)} draws archived in {header}.h5")
    return result, diag


def systematics_summary(header, label):
    """Signed coil drift + boundary RMS -- a bias is a non-zero MEAN."""
    import h5py, json as _json
    from scipy.spatial import cKDTree
    with h5py.File(header + '.h5', 'r') as f:
        g = f['scan/0']; bl = g['_baseline']
        bln = [(n.decode() if isinstance(n, bytes) else str(n))
               for n in bl['coil_names'][()]]
        base = dict(zip(bln, np.array(bl['coil_currents [A]'])))
        draws = sorted(int(k) for k in g if k.isdigit())
        def drift(coil):
            out = []
            for i in draws:
                nm = _json.loads(g[str(i)].attrs['coil_names'])
                v  = np.array(g[str(i)]['coil_currents [A]'])
                out.append(100*(dict(zip(nm, v))[coil] - base[coil])
                           / abs(base[coil]))
            return np.array(out)
        ref = np.asarray(bl['recon_lcfs_ref'][()]) if 'recon_lcfs_ref' in bl else None
        rms = []
        for i in draws:
            gi = g[str(i)]
            if ref is not None and 'perturbed_lcfs_ref' in gi:
                p = np.asarray(gi['perturbed_lcfs_ref'][()])
                d, _ = cKDTree(p).query(ref)
                rms.append(np.sqrt((d**2).mean())*1e3)
        inspec = sum(1 for i in draws if bool(g[str(i)].attrs.get('in_spec')))
        f9a, f9b = drift('F9A'), drift('F9B')
        fco = [n for n in bln if n.startswith('F') and n not in ('F9A', 'F9B')]
        allF = np.concatenate([drift(c) for c in fco]) if fco else np.array([0.0])
    rms = np.array(rms) if rms else np.array([np.nan])
    print(f"[{label}]  draws={len(draws)}  in-spec={inspec}/{len(draws)}")
    print(f"   boundary RMS [mm] : median {np.nanmedian(rms):.4f}  max {np.nanmax(rms):.4f}")
    print(f"   F9A drift %       : mean {f9a.mean():+.4f}  std {f9a.std():.4f}  (mean~0 => no bias)")
    print(f"   F9B drift %       : mean {f9b.mean():+.4f}  std {f9b.std():.4f}")
    print(f"   non-VSC F drift % : mean {allF.mean():+.5f}  std {allF.std():.4f}")
    return dict(label=label, n=len(draws), inspec=inspec,
                bnd_rms_med=float(np.nanmedian(rms)),
                bnd_rms_max=float(np.nanmax(rms)),
                f9a_mean=float(f9a.mean()), f9b_mean=float(f9b.mean()),
                allF_mean=float(allF.mean()))

summaries = {}

## 6. Mode 1 — j_phi pinned, σ=0  (no-systematic floor)

In [ ]:
m = MODES[0]
print('=== MODE 1:', m[3], '===')
res, diag = run_mode(m[0], m[1], m[2])

In [ ]:
# traces, coil currents, and boundary-point traces for this mode
_h = MODES[0][0]
plot_traces(_h, scan_value='all')
plot_coil_currents(_h, scan_val=0)
plot_boundary_point_traces(_h)
summaries[MODES[0][3].split(' (')[0]] = systematics_summary(_h, MODES[0][3])
import matplotlib.pyplot as plt; plt.show()

## 7. Mode 2 — j_phi pinned, IDA-like σ

In [ ]:
m = MODES[1]
print('=== MODE 2:', m[3], '===')
res, diag = run_mode(m[0], m[1], m[2])

In [ ]:
# traces, coil currents, and boundary-point traces for this mode
_h = MODES[1][0]
plot_traces(_h, scan_value='all')
plot_coil_currents(_h, scan_val=0)
plot_boundary_point_traces(_h)
summaries[MODES[1][3].split(' (')[0]] = systematics_summary(_h, MODES[1][3])
import matplotlib.pyplot as plt; plt.show()

## 8. Mode 3 — production (free j_phi + IDA σ)

In [ ]:
m = MODES[2]
print('=== MODE 3:', m[3], '===')
res, diag = run_mode(m[0], m[1], m[2])

In [ ]:
# traces, coil currents, and boundary-point traces for this mode
_h = MODES[2][0]
plot_traces(_h, scan_value='all')
plot_coil_currents(_h, scan_val=0)
plot_boundary_point_traces(_h)
summaries[MODES[2][3].split(' (')[0]] = systematics_summary(_h, MODES[2][3])
import matplotlib.pyplot as plt; plt.show()

## 9. Systematics verdict

In [ ]:
# ---- consolidated systematics verdict ----
print('mode'.ljust(38), 'in-spec', 'bnd_rms_med[mm]', 'F9A_mean%', 'F9B_mean%')
for k, s in summaries.items():
    print(s['label'].ljust(38),
          f"{s['inspec']}/{s['n']}".ljust(7),
          f"{s['bnd_rms_med']:.4f}".ljust(15),
          f"{s['f9a_mean']:+.4f}".ljust(9),
          f"{s['f9b_mean']:+.4f}")

# Mode 1 (pinned, sigma=0) is the strict no-systematic floor: every draw
# should reproduce the baseline, so boundary RMS and coil drift must be ~0.
s0 = summaries.get('pinned, sigma=0')
if s0 is not None:
    ok = (s0['bnd_rms_med'] < 0.5 and abs(s0['f9a_mean']) < 0.1
          and abs(s0['f9b_mean']) < 0.1 and abs(s0['allF_mean']) < 0.1)
    print()
    print('NO-SYSTEMATIC FLOOR (pinned sigma=0):',
          'PASS' if ok else 'CHECK -- non-zero offset detected')
# Modes 2 & 3: the signed coil-drift MEANS should sit near zero (symmetric
# scatter); a consistently non-zero mean would indicate a systematic bias.